In [ ]:
"""
==========================================================================
  Qwen2-VL-2B VQA 챌린지 파이프라인 v2 — 전면 재설계
==========================================================================
  
  [해결한 문제]
  1. Mode Collapse: 기존 코드는 labels에서 프롬프트 부분도 Loss 계산에
     포함시켜, 모델이 "질문을 잘 복사하는 것"을 학습함 → 정답 토큰에만
     Loss를 계산하도록 수정
  2. generate() 기반 추론: 토큰 생성 과정에서 노이즈가 끼어 a/b/c/d 외
     답변 발생 → Logit Scoring으로 강제 객관식 추론
  3. max_steps 기반 학습: 데이터셋을 충분히 보지 못함 → Epoch 기반으로 변경
  
  [추가 성능 부스팅]
  4. Multi-seed 앙상블: 서로 다른 시드로 3회 학습 → 추론 시 Logit 평균
  5. Dev set 검증: majority vote 기반 dev.csv로 정확도 측정
  6. Cosine LR 스케줄러 + Warmup
  
  사용법:
    python pipeline_v2.py --mode train       # 학습
    python pipeline_v2.py --mode inference   # 추론 (submission 생성)
    python pipeline_v2.py --mode ensemble    # 멀티시드 앙상블 추론
    python pipeline_v2.py --mode validate    # dev.csv로 정확도 검증
==========================================================================
"""

import os
import re
import gc
import argparse
import numpy as np
import torch
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoProcessor,
    Qwen2VLForConditionalGeneration,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel


# ══════════════════════════════════════════════
# 설정 (본인 환경에 맞게 수정)
# ══════════════════════════════════════════════
# --- 코랩 환경 ---
#BASE_DIR = Path("/content/drive/MyDrive/2026-ssafy-15-2-ai_dataset/2026-ssafy-15-2-ai_dataset")
# --- 로컬 환경 (필요 시 주석 해제) ---
BASE_DIR = Path(r"C:\Users\SSAFY\Downloads\2026-ssafy-ai-15-2")

TRAIN_CSV   = BASE_DIR / "train_final.csv"          # 원본 또는 증강된 train CSV
DEV_CSV     = BASE_DIR / "dev.csv"
TEST_CSV    = BASE_DIR / "test.csv"
MODEL_ID    = "Qwen/Qwen2-VL-2B-Instruct"
OUTPUT_DIR  = BASE_DIR / "vqa_lora_v2"
OUTPUT_CSV  = BASE_DIR / "submission_final.csv"

# 학습 하이퍼파라미터
NUM_EPOCHS       = 3
BATCH_SIZE_TRAIN = 2
GRAD_ACCUM       = 8
LEARNING_RATE    = 2e-4
LORA_R           = 16
LORA_ALPHA       = 32
BATCH_SIZE_INFER = 4   # 추론 배치 사이즈 (OOM 시 2로)

# 앙상블 시드 목록
ENSEMBLE_SEEDS = [42, 123, 777]


# ══════════════════════════════════════════════
# 공통 유틸리티
# ══════════════════════════════════════════════
def build_prompt(row: pd.Series) -> str:
    """학습과 추론에서 동일한 프롬프트 포맷을 보장"""
    return (
        "You are a recycling classification expert. "
        "Look at the image carefully and choose the single best answer.\n"
        f"Question: {row['question']}\n"
        f"Options:\n"
        f"a) {row['a']}\n"
        f"b) {row['b']}\n"
        f"c) {row['c']}\n"
        f"d) {row['d']}\n"
        f"Answer with only one letter (a, b, c, or d):"
    )


def get_bnb_config() -> BitsAndBytesConfig:
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )


def get_answer_token_ids(processor) -> dict:
    """a, b, c, d 각각의 토큰 ID를 미리 조회"""
    tokenizer = processor.tokenizer
    ids = {}
    for ch in ["a", "b", "c", "d"]:
        token_ids = tokenizer.encode(ch, add_special_tokens=False)
        ids[ch] = token_ids[-1]  # 마지막 토큰 (서브워드 대응)
    return ids


# ══════════════════════════════════════════════
# 1. 데이터셋
# ══════════════════════════════════════════════
class VQATrainDataset(Dataset):
    """학습용: 메시지 + 정답을 함께 반환"""
    def __init__(self, df: pd.DataFrame, base_dir: Path):
        self.df = df.reset_index(drop=True)
        self.base_dir = base_dir

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx):
        if isinstance(idx, (list, slice)):
            indices = range(len(self))[idx] if isinstance(idx, slice) else idx
            return [self.__getitem__(i) for i in indices]

        row = self.df.iloc[idx]
        image_path = str(self.base_dir / row["path"])
        prompt = build_prompt(row)
        answer = str(row["answer"]).strip().lower()

        messages = [
            {"role": "user", "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": prompt},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": answer},
            ]},
        ]
        return {"messages": messages, "image_path": image_path}


class VQATestDataset(Dataset):
    """추론용: 이미지 + 프롬프트만 반환 (정답 없음)"""
    def __init__(self, df: pd.DataFrame, base_dir: Path):
        self.df = df.reset_index(drop=True)
        self.base_dir = base_dir

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx) -> dict:
        row = self.df.iloc[idx]
        image_path = str(self.base_dir / row["path"])
        prompt = build_prompt(row)

        messages = [
            {"role": "user", "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": prompt},
            ]},
        ]
        return {"id": row["id"], "messages": messages, "image_path": image_path}


# ══════════════════════════════════════════════
# 2. DataCollator — 핵심: Loss Masking
# ══════════════════════════════════════════════
# [수정된 DataCollator: 동적 길이 텐서 변환 에러 완벽 해결 및 정밀 마스킹]
class VQADataCollatorWithMasking:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        if isinstance(features, dict):
            features = [features]

        texts_with_answer = []
        texts_without_answer = []
        images = []

        for feature in features:
            if isinstance(feature, list):
                feature = feature[0]

            messages = feature["messages"]

            # 1. 정답 포함 전체 텍스트
            full_text = self.processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            texts_with_answer.append(full_text)

            # 2. 정답 제외 프롬프트 텍스트
            user_only = [m for m in messages if m["role"] != "assistant"]
            prompt_text = self.processor.apply_chat_template(
                user_only, tokenize=False, add_generation_prompt=True
            )
            texts_without_answer.append(prompt_text)

            img = Image.open(feature["image_path"]).convert("RGB")
            images.append(img)

        # ── 전체 시퀀스 토크나이즈 (학습용 Tensor 변환) ──
        inputs = self.processor(
            text=texts_with_answer,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        # ── 프롬프트 길이 측정용 토크나이즈 (List 반환으로 에러 방지) ──
        prompt_inputs = self.processor(
            text=texts_without_answer,
            images=images,
            padding=False,
            return_tensors=None,  # [핵심] 길이가 다르므로 텐서 변환을 포기하고 리스트로 받음
        )

        labels = inputs["input_ids"].clone()
        pad_token_id = self.processor.tokenizer.pad_token_id

        # 패딩 토큰은 Loss 계산 제외 (-100)
        if pad_token_id is not None:
            labels[labels == pad_token_id] = -100

        # ── 샘플별 프롬프트 정밀 마스킹 ──
        for i, prompt_ids in enumerate(prompt_inputs["input_ids"]):
            p_len = len(prompt_ids)
            
            if pad_token_id is not None:
                # 패딩이 아닌 실제 토큰의 시작 인덱스 탐색 (Left/Right 패딩 무관하게 작동)
                non_pad_idx = (inputs["input_ids"][i] != pad_token_id).nonzero(as_tuple=True)[0]
                start_idx = non_pad_idx[0].item() if len(non_pad_idx) > 0 else 0
            else:
                start_idx = 0
            
            # 모델이 '질문' 부분은 무시하고 '정답'에만 집중하도록 마스킹
            labels[i, start_idx : start_idx + p_len] = -100

        inputs["labels"] = labels
        return inputs

# ══════════════════════════════════════════════
# 3. 학습 (Training)
# ══════════════════════════════════════════════
def train(seed: int = 42, output_suffix: str = ""):
    """QLoRA 파인튜닝 — Loss Masking + Epoch 기반 + Cosine LR"""
    torch.cuda.empty_cache()
    torch.manual_seed(seed)

    suffix = f"_seed{seed}" if output_suffix == "" else output_suffix
    save_dir = OUTPUT_DIR / f"final_lora{suffix}"

    print(f"\n{'='*60}")
    print(f"  학습 시작 (seed={seed})")
    print(f"  저장 경로: {save_dir}")
    print(f"{'='*60}\n")

    # ── 데이터 로드 ──
    print("[1/4] 데이터 로드 중...")
    df = pd.read_csv(TRAIN_CSV).dropna(subset=["question", "answer"])
    train_df = df.sample(frac=0.95, random_state=seed)
    eval_df = df.drop(train_df.index)

    train_dataset = VQATrainDataset(train_df, BASE_DIR)
    eval_dataset = VQATrainDataset(eval_df, BASE_DIR)
    print(f"   Train: {len(train_dataset):,}건 / Eval: {len(eval_dataset):,}건")

    # ── 프로세서 & 콜레이터 ──
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    data_collator = VQADataCollatorWithMasking(processor)

    # ── 모델 로드 (4-bit QLoRA) ──
    print("[2/4] 모델 로드 중 (4-bit 양자화)...")
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=get_bnb_config(), device_map="auto"
    )
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # ── TrainingArguments ──
    print("[3/4] 학습 설정 구성...")
    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR / f"checkpoints{suffix}"),
        num_train_epochs=NUM_EPOCHS,           # ★ Epoch 기반
        per_device_train_batch_size=BATCH_SIZE_TRAIN,
        gradient_accumulation_steps=GRAD_ACCUM,
        optim="paged_adamw_32bit",
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",            # ★ Cosine 스케줄러
        warmup_ratio=0.05,                     # ★ Warmup 5%
        fp16=True,
        logging_steps=20,
        save_strategy="epoch",
        eval_strategy="epoch",
        save_total_limit=2,
        remove_unused_columns=False,
        logging_first_step=True,
        report_to="none",
        dataloader_num_workers=0,
        seed=seed,
    )

    # ── Trainer ──
    print("[4/4] 파인튜닝 시작...")
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
    )

    trainer.train()
    trainer.model.save_pretrained(str(save_dir))
    print(f"\n✅ 학습 완료! 어댑터 저장: {save_dir}")

    # 메모리 해제
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()


# ══════════════════════════════════════════════
# 4. 추론 — Logit Scoring (강제 객관식)
# ══════════════════════════════════════════════
def logit_inference(
    lora_path: Path,
    test_df: pd.DataFrame,
    return_logits: bool = False,
) -> pd.DataFrame | tuple:
    """
    [generate() 대신 Logit Scoring]
    
    1. 프롬프트를 모델에 forward pass
    2. 마지막 토큰 위치의 logits에서 a/b/c/d 토큰 ID만 추출
    3. argmax → 강제로 a/b/c/d 중 하나 선택
    
    이 방식은 generate()보다 빠르고, 반드시 유효한 답만 나옴
    """
    torch.cuda.empty_cache()

    print(f"  모델 로드: {lora_path}")
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    base_model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=get_bnb_config(), device_map="auto"
    )
    model = PeftModel.from_pretrained(base_model, str(lora_path))
    model.eval()

    # a, b, c, d 토큰 ID 조회
    answer_ids = get_answer_token_ids(processor)
    id_to_letter = {v: k for k, v in answer_ids.items()}
    answer_token_ids = [answer_ids["a"], answer_ids["b"], answer_ids["c"], answer_ids["d"]]
    print(f"  토큰 ID 매핑: {answer_ids}")

    test_dataset = VQATestDataset(test_df, BASE_DIR)
    results = []
    all_logits = []  # 앙상블용

    for start_idx in tqdm(range(0, len(test_dataset), BATCH_SIZE_INFER), desc="  추론"):
        end_idx = min(start_idx + BATCH_SIZE_INFER, len(test_dataset))
        batch_items = [test_dataset[i] for i in range(start_idx, end_idx)]

        batch_ids = [item["id"] for item in batch_items]
        batch_texts = []
        batch_images = []

        for item in batch_items:
            text = processor.apply_chat_template(
                item["messages"], tokenize=False, add_generation_prompt=True
            )
            batch_texts.append(text)
            batch_images.append(Image.open(item["image_path"]).convert("RGB"))

        inputs = processor(
            text=batch_texts,
            images=batch_images,
            padding=True,
            return_tensors="pt",
        ).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs)

        # ── Logit Scoring ──
        # 각 샘플의 마지막 유효 토큰 위치에서 logit 추출
        for i, sample_id in enumerate(batch_ids):
            # attention_mask에서 마지막 유효 위치 찾기
            attn = inputs["attention_mask"][i]
            last_pos = attn.sum().item() - 1

            # 해당 위치의 logits에서 a/b/c/d만 추출
            logits_at_last = outputs.logits[i, last_pos]
            choice_logits = torch.tensor([logits_at_last[tid].item() for tid in answer_token_ids])

            if return_logits:
                all_logits.append(choice_logits.numpy())

            # argmax → 정답
            best_idx = choice_logits.argmax().item()
            answer = ["a", "b", "c", "d"][best_idx]
            results.append({"id": sample_id, "answer": answer})

        del inputs, outputs
        torch.cuda.empty_cache()

    # 메모리 해제
    del model, base_model
    gc.collect()
    torch.cuda.empty_cache()

    result_df = pd.DataFrame(results)

    if return_logits:
        return result_df, np.array(all_logits)
    return result_df


# ══════════════════════════════════════════════
# 5. 단일 모델 추론 + 제출 파일 생성
# ══════════════════════════════════════════════
def inference():
    """단일 LoRA 어댑터로 추론 → submission_final.csv"""
    print("\n" + "=" * 60)
    print("  단일 모델 추론")
    print("=" * 60)

    test_df = pd.read_csv(TEST_CSV)
    print(f"  테스트 데이터: {len(test_df):,}건\n")

    # 기본 시드(42) 어댑터 사용
    lora_path = OUTPUT_DIR / "final_lora_seed42"
    if not lora_path.exists():
        lora_path = OUTPUT_DIR / "final_lora"  # fallback

    result_df = logit_inference(lora_path, test_df)
    result_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    print(f"\n✅ 저장 완료: {OUTPUT_CSV}")
    print(f"   총 {len(result_df):,}건")
    print(f"\n   [답변 분포]")
    for ans, cnt in result_df["answer"].value_counts().sort_index().items():
        print(f"   {ans}: {cnt:,}건 ({cnt/len(result_df)*100:.1f}%)")


# ══════════════════════════════════════════════
# 6. 멀티시드 앙상블 추론
# ══════════════════════════════════════════════
def ensemble_inference():
    """
    여러 시드로 학습한 모델들의 Logit을 평균내어 최종 답변 결정.
    
    단일 모델 대비 일반적으로 1~3%p 정확도 향상 기대.
    """
    print("\n" + "=" * 60)
    print("  멀티시드 앙상블 추론")
    print("=" * 60)

    test_df = pd.read_csv(TEST_CSV)
    print(f"  테스트 데이터: {len(test_df):,}건")

    all_logits = []

    for seed in ENSEMBLE_SEEDS:
        lora_path = OUTPUT_DIR / f"final_lora_seed{seed}"
        if not lora_path.exists():
            print(f"  ⚠️ {lora_path} 없음, 건너뜀")
            continue

        print(f"\n  [Seed {seed}] 추론 중...")
        _, logits = logit_inference(lora_path, test_df, return_logits=True)
        all_logits.append(logits)
        print(f"  [Seed {seed}] 완료 (logits shape: {logits.shape})")

    if not all_logits:
        print("❌ 앙상블할 모델이 없습니다. 먼저 --mode train으로 학습하세요.")
        return

    # ── Logit 평균 앙상블 ──
    mean_logits = np.mean(all_logits, axis=0)  # (N, 4)
    final_answers = [["a", "b", "c", "d"][i] for i in mean_logits.argmax(axis=1)]

    result_df = pd.DataFrame({
        "id": test_df["id"],
        "answer": final_answers,
    })

    ensemble_csv = BASE_DIR / "submission_ensemble.csv"
    result_df.to_csv(ensemble_csv, index=False, encoding="utf-8-sig")

    print(f"\n✅ 앙상블 결과 저장: {ensemble_csv}")
    print(f"   사용 모델 수: {len(all_logits)}")
    print(f"   총 {len(result_df):,}건")
    print(f"\n   [답변 분포]")
    for ans, cnt in result_df["answer"].value_counts().sort_index().items():
        print(f"   {ans}: {cnt:,}건 ({cnt/len(result_df)*100:.1f}%)")


# ══════════════════════════════════════════════
# 7. Dev 검증 (정확도 측정)
# ══════════════════════════════════════════════
def validate():
    """dev.csv의 majority vote 정답 대비 정확도 측정"""
    print("\n" + "=" * 60)
    print("  Dev 검증")
    print("=" * 60)

    dev_df = pd.read_csv(DEV_CSV)

    # Majority vote로 정답 생성
    def majority_vote(row):
        votes = [row[f"answer{i}"] for i in range(1, 6)]
        return Counter(votes).most_common(1)[0][0]

    dev_df["answer"] = dev_df.apply(majority_vote, axis=1)
    print(f"  Dev 데이터: {len(dev_df):,}건 (majority vote 적용)\n")

    lora_path = OUTPUT_DIR / "final_lora_seed42"
    if not lora_path.exists():
        lora_path = OUTPUT_DIR / "final_lora"

    result_df = logit_inference(lora_path, dev_df)

    # 정확도 계산
    correct = (result_df["answer"].values == dev_df["answer"].values).sum()
    total = len(dev_df)
    accuracy = correct / total * 100

    print(f"\n{'='*40}")
    print(f"  Dev Accuracy: {correct}/{total} = {accuracy:.2f}%")
    print(f"{'='*40}")

    # 오답 분석
    wrong_mask = result_df["answer"].values != dev_df["answer"].values
    if wrong_mask.sum() > 0:
        print(f"\n  [오답 분포]")
        wrong_preds = result_df.loc[wrong_mask, "answer"]
        for ans, cnt in wrong_preds.value_counts().sort_index().items():
            print(f"   예측 '{ans}': {cnt}건")


# ══════════════════════════════════════════════
# 메인
# ══════════════════════════════════════════════
if __name__ == "__main__":
    # 메모리 초기화
    torch.cuda.empty_cache()
    
    # 1단계: 모델 학습 (원하는 작업의 주석을 해제하여 하나씩 실행하십시오)
    train(seed=42)
    
    # 2단계: 학습이 완료된 후, 로컬 Dev 셋으로 정확도 자체 검증
    #validate()
    
    # 3단계: 최종 제출용 CSV 생성
    #inference()



  학습 시작 (seed=42)
  저장 경로: C:\Users\SSAFY\Downloads\2026-ssafy-ai-15-2\vqa_lora_v2\final_lora_seed42

[1/4] 데이터 로드 중...
   Train: 57,416건 / Eval: 3,022건
[2/4] 모델 로드 중 (4-bit 양자화)...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

c:\Users\SSAFY\AppData\Local\Programs\Python\Python311\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 4,358,144 || all params: 2,213,343,744 || trainable%: 0.1969
[3/4] 학습 설정 구성...
[4/4] 파인튜닝 시작...


c:\Users\SSAFY\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\_dynamo\eval_frame.py:1272: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss
